# Feature Engineering
## 03 — Menyiapkan Riwayat sebagai Input dan Minggu Depan sebagai Target

**Proyek:** Prediksi Muka Air Tanah Satu Minggu ke Depan · **Studi kasus:** Drenthe, Belanda · **Penyusun:** Rasen

Pada notebook pertama, saya menyiapkan kalender mingguan dan menandai kelengkapan pengamatan. Pada notebook kedua, saya mengeksplorasi pola hanya pada data pengembangan serta menyisihkan data mulai 1 Januari 2017 sebagai holdout.

Sekarang saya mengubah hasil pengolahan tersebut menjadi tabel yang dapat dipakai untuk supervised learning. Satu baris akan memuat informasi yang diketahui sampai akhir minggu tertentu dan nilai yang ingin diprediksi pada minggu berikutnya.

**Alur:** membaca data → membuat fitur historis → menyiapkan target dan baseline → memeriksa ketersediaan sampel → membagi berdasarkan tanggal target → menyimpan data untuk modeling.

### Cara menjalankan

Simpan notebook ini di `notebooks`, lalu jalankan dari atas ke bawah setelah notebook 01 dan 02 selesai. Kode membaca `data/processed/netherlands_weekly.csv` dan `outputs/eda/konfigurasi_eksperimen.json`. Tidak perlu mengganti versi Python yang sudah berhasil dipakai.

Notebook ini menulis file baru di `data/processed` serta dokumentasi di `outputs/features`. File sumber tidak diubah. Jika dijalankan ulang, file hasil dengan nama sama akan diperbarui.


## 1. Memahami Fitur dan Target dengan Contoh

Misalnya saya selesai mengamati minggu yang berakhir pada 9 Januari. Saya ingin memprediksi rata-rata head pada minggu yang berakhir 16 Januari. Informasi tanggal 16 Januari belum boleh dipakai sebagai input.

| Bagian | Makna | Contoh posisi waktu |
| --- | --- | --- |
| `forecast_date` | Akhir minggu ketika prediksi dibuat | 9 Januari |
| `head_t` | Head terbaru yang sudah tersedia | Minggu berakhir 9 Januari |
| `head_lag_1w` | Head satu minggu sebelum waktu prediksi | Minggu berakhir 2 Januari |
| `target_date` | Akhir minggu yang ingin diprediksi | 16 Januari |
| `target_head_m` | Rata-rata head pada minggu target | Nilai yang dipelajari/dievaluasi, bukan input |

`X` adalah kumpulan fitur, sedangkan `y` adalah target. Pemisahan ini penting supaya kita tidak tanpa sengaja memasukkan jawaban ke dalam input model.

Ini tetap eksperimen historis. Penggunaan data cuaca minggu berjalan mengasumsikan data tersebut sudah tersedia pada waktu prediksi; keterlambatan penerbitan arsip cuaca belum dimodelkan.


In [1]:
from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd

kandidat = [Path.cwd(), *Path.cwd().parents,
    Path.home()/"Downloads"/"Rasen_Portofolio"/"02_Projects"/"groundwater_forecasting"]
PROJECT_DIR = next((p for p in kandidat if (p/"data/processed/netherlands_weekly.csv").is_file()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError("File mingguan belum ditemukan. Jalankan notebook 01 dari folder proyek.")

file_mingguan = PROJECT_DIR/"data/processed/netherlands_weekly.csv"
file_konfigurasi = PROJECT_DIR/"outputs/eda/konfigurasi_eksperimen.json"
if not file_konfigurasi.is_file():
    raise FileNotFoundError("Konfigurasi EDA belum ada. Jalankan notebook 02 sampai selesai.")
konfigurasi_eda = json.loads(file_konfigurasi.read_text(encoding="utf-8"))
BATAS_HOLDOUT = pd.Timestamp(konfigurasi_eda["batas_holdout_tanggal_target"])
assert konfigurasi_eda["horizon_minggu"] == 1
assert konfigurasi_eda["frekuensi"] == "W-SUN"

mingguan = pd.read_csv(file_mingguan, parse_dates=["date"]).sort_values("date").set_index("date")
kolom_cuaca = ["rainfall_mm", "temperature_mean_c", "temperature_min_c", "temperature_max_c",
    "sea_level_pressure_hpa", "relative_humidity_pct", "wind_speed_ms",
    "global_radiation_wm2", "potential_evaporation_mm"]
kolom_wajib = ["groundwater_head_m", "head_obs_count", "weather_obs_count", "week_valid", *kolom_cuaca]
if not set(kolom_wajib).issubset(mingguan.columns):
    raise ValueError("Kolom file mingguan tidak sesuai notebook 01.")
assert not mingguan.empty and not mingguan.index.hasnans
assert not mingguan.index.duplicated().any()
assert (mingguan.index.dayofweek == 6).all()
assert mingguan.index.to_series().diff().dropna().eq(pd.Timedelta(days=7)).all()
penanda = mingguan["week_valid"].astype(str).str.lower().str.strip()
assert penanda.isin(["true", "false"]).all()
mingguan["week_valid"] = penanda.eq("true")
assert mingguan["week_valid"].eq((mingguan["head_obs_count"] >= 4) & (mingguan["weather_obs_count"] == 7)).all()
assert mingguan.loc[~mingguan["week_valid"], "groundwater_head_m"].isna().all()
assert mingguan.loc[mingguan["week_valid"], ["groundwater_head_m", *kolom_cuaca]].notna().all().all()
mingguan.loc[mingguan["weather_obs_count"] < 7, kolom_cuaca] = np.nan

PROCESSED_DIR = PROJECT_DIR/"data/processed"
FEATURE_DIR = PROJECT_DIR/"outputs/features"
FEATURE_DIR.mkdir(parents=True, exist_ok=True)
print("Minggu kalender:", len(mingguan))
print("Batas holdout berdasarkan tanggal target:", BATAS_HOLDOUT.date())


Minggu kalender: 1092
Batas holdout berdasarkan tanggal target: 2017-01-01


## 2. Membuat Fitur Historis pada Kalender Lengkap

Saya mencoba tiga kelompok fitur. Kelompok pertama hanya menggunakan riwayat head. Kelompok kedua menambahkan cuaca, dan kelompok ketiga menambahkan posisi musim. Perbandingan ini membantu menjawab apakah tambahan informasi memberi manfaat, bukan sekadar memperbanyak kolom.

| Kelompok fitur | Isi | Jumlah |
| --- | --- | ---: |
| `head_only` | Head saat ini; lag 1, 2, 4 minggu; rata-rata 4 minggu; perubahan 1 minggu | 6 |
| `head_weather` | Fitur head + 9 cuaca saat ini + total hujan/evaporasi 4 dan 8 minggu | 19 |
| `head_weather_season` | Kelompok sebelumnya + sinus dan kosinus musim | 21 |

`shift(1)` mengambil nilai dari minggu sebelumnya. `rolling(4, min_periods=4)` merangkum empat minggu sampai minggu saat ini dan hanya menghasilkan nilai jika seluruh empat pengamatan tersedia. Jendelanya tidak terpusat dan tidak membaca masa depan.

Saya belum membuang baris kosong. Kalau kalender dipadatkan dulu, lag satu baris bisa melompat melewati celah berbulan-bulan. Untuk fitur musim, posisi kalender diubah menjadi sinus–kosinus sehingga akhir dan awal tahun tetap berdekatan.


In [2]:
def buat_fitur(tabel):
    """Semua fitur hanya memakai informasi pada atau sebelum akhir minggu t."""
    fitur = pd.DataFrame(index=tabel.index)
    head = tabel["groundwater_head_m"]
    fitur["head_t"] = head
    for lag in [1, 2, 4]:
        fitur[f"head_lag_{lag}w"] = head.shift(lag)
    fitur["head_mean_4w"] = head.rolling(4, min_periods=4).mean()
    fitur["head_change_1w"] = head.diff()

    for kolom in kolom_cuaca:
        fitur[f"{kolom}_t"] = tabel[kolom]
    for jendela in [4, 8]:
        fitur[f"rainfall_sum_{jendela}w"] = tabel["rainfall_mm"].rolling(jendela, min_periods=jendela).sum()
        fitur[f"evaporation_sum_{jendela}w"] = tabel["potential_evaporation_mm"].rolling(jendela, min_periods=jendela).sum()

    panjang_tahun = np.where(tabel.index.is_leap_year, 366, 365)
    fase = 2 * np.pi * (tabel.index.dayofyear - 1) / panjang_tahun
    fitur["season_sin"] = np.sin(fase)
    fitur["season_cos"] = np.cos(fase)
    return fitur

fitur = buat_fitur(mingguan)
fitur_head = ["head_t", "head_lag_1w", "head_lag_2w", "head_lag_4w", "head_mean_4w", "head_change_1w"]
fitur_cuaca = [f"{kolom}_t" for kolom in kolom_cuaca] + [
    "rainfall_sum_4w", "evaporation_sum_4w", "rainfall_sum_8w", "evaporation_sum_8w"]
kelompok_fitur = {
    "head_only": fitur_head,
    "head_weather": fitur_head + fitur_cuaca,
    "head_weather_season": fitur_head + fitur_cuaca + ["season_sin", "season_cos"]
}
semua_fitur = kelompok_fitur["head_weather_season"]
assert len(semua_fitur) == len(set(semua_fitur)) == 21
print({nama: len(kolom) for nama, kolom in kelompok_fitur.items()})
display(fitur.loc[fitur.index < BATAS_HOLDOUT, fitur_head].head(10).round(3))


{'head_only': 6, 'head_weather': 19, 'head_weather_season': 21}


,head_t,head_lag_1w,head_lag_2w,head_lag_4w,head_mean_4w,head_change_1w
date,,,,,,
2000-01-02,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-09,11.239,NaN,NaN,NaN,NaN,NaN
2000-01-16,11.213,11.239,NaN,NaN,NaN,-0.026
2000-01-23,11.221,11.213,11.239,NaN,NaN,0.009
2000-01-30,11.224,11.221,11.213,NaN,11.224,0.003
2000-02-06,11.244,11.224,11.221,11.239,11.226,0.020
2000-02-13,11.249,11.244,11.224,11.213,11.235,0.004
2000-02-20,11.253,11.249,11.244,11.221,11.242,0.004
2000-02-27,11.227,11.253,11.249,11.224,11.243,-0.026


## 3. Menyiapkan Target dan Dua Baseline

Target dibuat dengan menggeser head satu minggu ke depan. Pada baris minggu `t`, `target_head_m` berisi head minggu `t+1`. Target yang tidak tersedia tetap kosong; tidak ada interpolasi target.

**Persistence** memperkirakan minggu depan sama dengan head minggu ini. **Seasonal naïve** memperkirakan minggu depan memakai nilai 52 minggu sebelum minggu target, yaitu `head(t−51)`. Angka 51 muncul karena targetnya sudah berada satu minggu di depan waktu prediksi.

Nilai seasonal naïve bisa kosong ketika riwayatnya tidak tersedia. Baseline ini dinilai pada tanggal yang memiliki nilai pembanding, lalu model lain juga dibandingkan pada irisan tanggal yang sama. Kekosongan baseline tidak boleh diisi dengan target sebenarnya.


In [3]:
supervised = fitur.copy()
supervised.index.name = "forecast_date"
supervised["target_date"] = supervised.index + pd.Timedelta(weeks=1)
supervised["target_head_m"] = mingguan["groundwater_head_m"].shift(-1)
supervised["baseline_persistence"] = mingguan["groundwater_head_m"]
supervised["baseline_seasonal_52w"] = mingguan["groundwater_head_m"].shift(51)
supervised["forecast_week_valid"] = mingguan["week_valid"]
supervised["target_available"] = supervised["target_head_m"].notna()
supervised["features_complete"] = supervised[semua_fitur].notna().all(axis=1)
supervised["sample_valid"] = (supervised["forecast_week_valid"]
    & supervised["target_available"] & supervised["features_complete"])

# Target dan baseline hanya metadata; keduanya tidak masuk daftar X.
assert not set(semua_fitur).intersection({"target_head_m", "target_date", "baseline_persistence", "baseline_seasonal_52w"})
assert (supervised["target_date"] > supervised.index).all()
display(supervised.loc[supervised["target_date"] < BATAS_HOLDOUT,
    ["head_t", "target_date", "target_head_m", "baseline_persistence", "baseline_seasonal_52w", "sample_valid"]].head(12))


,head_t,target_date,target_head_m,baseline_persistence,baseline_seasonal_52w,sample_valid
forecast_date,,,,,,
2000-01-02,NaN,2000-01-09,11.238571,NaN,NaN,False
2000-01-09,11.238571,2000-01-16,11.212857,11.238571,NaN,False
2000-01-16,11.212857,2000-01-23,11.221429,11.212857,NaN,False
2000-01-23,11.221429,2000-01-30,11.224286,11.221429,NaN,False
2000-01-30,11.224286,2000-02-06,11.244286,11.224286,NaN,False
2000-02-06,11.244286,2000-02-13,11.248571,11.244286,NaN,False
2000-02-13,11.248571,2000-02-20,11.252857,11.248571,NaN,False
2000-02-20,11.252857,2000-02-27,11.227143,11.252857,NaN,False
2000-02-27,11.227143,2000-03-05,11.250000,11.227143,NaN,True


## 4. Pemeriksaan Arah Waktu

Pemeriksaan berikut menggunakan dua cara. Pertama, saya memastikan pergeseran lag dan target sesuai dengan definisinya. Kedua, saya menghitung ulang fitur dari potongan data lama. Fitur pada potongan itu harus sama dengan fitur tanggal yang sama saat kalender lengkap dipakai.

Jika hasil berubah hanya karena data masa depan ditambahkan, berarti ada transformasi yang membaca informasi masa depan. Pemeriksaan ini berguna untuk mencegah kesalahan pada rolling, lag, atau ringkasan global.


In [4]:
pd.testing.assert_series_equal(fitur["head_lag_1w"],
    mingguan["groundwater_head_m"].shift(1), check_names=False)
pd.testing.assert_series_equal(supervised["target_head_m"],
    mingguan["groundwater_head_m"].shift(-1), check_names=False)
pd.testing.assert_series_equal(supervised["baseline_seasonal_52w"],
    mingguan["groundwater_head_m"].shift(51), check_names=False)

potongan = mingguan.loc[mingguan.index < pd.Timestamp("2010-01-01")]
if potongan.empty:
    raise ValueError("Data sebelum 2010 tidak ditemukan; periksa dataset proyek.")
pd.testing.assert_frame_equal(buat_fitur(potongan), fitur.loc[potongan.index])
angka_fitur = supervised[semua_fitur].to_numpy(dtype=float)
assert not np.isinf(angka_fitur).any()
print("Pemeriksaan lag, target, baseline musiman, dan fitur tanpa data masa depan: lolos.")


Pemeriksaan lag, target, baseline musiman, dan fitur tanpa data masa depan: lolos.


## 5. Memilih Sampel dan Membagi Berdasarkan Tanggal Target

Saya mempertahankan tabel kalender lengkap untuk audit. Untuk eksperimen, saya memilih baris yang mempunyai target dan seluruh 21 kandidat fitur. Semua kelompok fitur memakai baris yang sama, sehingga perbedaan skor tidak berasal dari perbedaan periode evaluasi.

Keputusan complete-case ini memang mengurangi cakupan. Model head-only sebenarnya dapat bekerja pada lebih banyak baris, tetapi eksperimen perbandingan menggunakan irisan bersama. Jumlah sampel yang gugur tetap dilaporkan.

Holdout dimulai ketika **tanggal target** mencapai 1 Januari 2017. Riwayat sebelum tanggal tersebut boleh digunakan untuk membuat prediksi holdout. Dalam evaluasi satu langkah ke depan, observasi minggu holdout yang sudah berlalu juga boleh menjadi lag untuk minggu selanjutnya. Ini bukan ramalan seluruh 2017–2020 sekaligus dari akhir 2016.

Kode hanya menampilkan jumlah dan rentang holdout, bukan skor atau hubungan antarvariabelnya.


In [5]:
mask_dev = supervised["target_date"] < BATAS_HOLDOUT
mask_holdout = supervised["target_date"] >= BATAS_HOLDOUT

def audit_sampel(nama, mask):
    bagian = supervised.loc[mask]
    return {
        "bagian": nama, "baris_kalender": len(bagian),
        "target_tersedia": int(bagian["target_available"].sum()),
        "fitur_lengkap": int(bagian["features_complete"].sum()),
        "sampel_layak": int(bagian["sample_valid"].sum()),
        "sampel_tidak_layak": int((~bagian["sample_valid"]).sum())
    }
audit = pd.DataFrame([audit_sampel("Pengembangan", mask_dev), audit_sampel("Holdout", mask_holdout)])
display(audit)

pengembangan = supervised.loc[mask_dev & supervised["sample_valid"]].copy()
holdout = supervised.loc[mask_holdout & supervised["sample_valid"]].copy()
assert not pengembangan.empty and not holdout.empty
assert pengembangan["target_date"].max() < BATAS_HOLDOUT <= holdout["target_date"].min()
assert pengembangan[semua_fitur + ["target_head_m"]].notna().all().all()
assert holdout[semua_fitur + ["target_head_m"]].notna().all().all()

for nama, tabel in [("Pengembangan", pengembangan), ("Holdout", holdout)]:
    print(nama, "—", len(tabel), "sampel; target", tabel["target_date"].min().date(),
          "sampai", tabel["target_date"].max().date())
print("Jumlah target tersedia dan fitur lengkap bisa tumpang tindih; kolom audit bukan kategori yang dijumlahkan.")


,bagian,baris_kalender,target_tersedia,fitur_lengkap,sampel_layak,sampel_tidak_layak
0,Pengembangan,886,827,807,804,82
1,Holdout,206,204,205,204,2


Pengembangan — 804 sampel; target 2000-03-05 sampai 2016-12-25
Holdout — 204 sampel; target 2017-01-01 sampai 2020-11-22
Jumlah target tersedia dan fitur lengkap bisa tumpang tindih; kolom audit bukan kategori yang dijumlahkan.


## 6. Menyimpan Data dan Definisi Fitur

Ada tiga keluaran data. Kalender lengkap dipakai untuk audit. Data pengembangan dipakai untuk modeling. Holdout disimpan terpisah agar tidak ikut terbaca saat pemilihan model.

File JSON menyimpan urutan kolom input, definisi prediksi, dan batas waktu. Model nanti membaca daftar fitur secara eksplisit, bukan mengambil semua kolom numerik yang berisiko mencakup target.

Saya belum melakukan scaling. Scaling mempelajari rata-rata dan simpangan baku sehingga baru dilakukan di dalam pipeline pada setiap bagian latihan.


In [6]:
file_keluaran = {
    "netherlands_supervised_calendar.csv": supervised,
    "netherlands_development.csv": pengembangan,
    "netherlands_holdout.csv": holdout
}
for nama, tabel in file_keluaran.items():
    tabel.reset_index().to_csv(PROCESSED_DIR/nama, index=False)

skema = {
    "version": 1, "horizon_weeks": 1, "frequency": "W-SUN",
    "holdout_target_start": BATAS_HOLDOUT.strftime("%Y-%m-%d"),
    "target_column": "target_head_m", "forecast_date_column": "forecast_date",
    "target_date_column": "target_date", "feature_groups": kelompok_fitur,
    "row_policy": "common complete cases for all feature groups; no target imputation",
    "forecast_policy": "rolling one-week-ahead with observed history; not multi-step recursion",
    "seasonal_baseline_shift_from_forecast": 51,
    "source_sha256": hashlib.sha256(file_mingguan.read_bytes()).hexdigest(),
    "development_sha256": hashlib.sha256((PROCESSED_DIR/"netherlands_development.csv").read_bytes()).hexdigest(),
    "holdout_sha256": hashlib.sha256((PROCESSED_DIR/"netherlands_holdout.csv").read_bytes()).hexdigest()
}
(PROCESSED_DIR/"feature_schema.json").write_text(json.dumps(skema, indent=2, ensure_ascii=False), encoding="utf-8")
audit.to_csv(FEATURE_DIR/"audit_sampel.csv", index=False)
print("Selesai. Tiga CSV dan feature_schema.json tersimpan di data/processed.")
print("Berikutnya buka 04_modeling.ipynb.")


Selesai. Tiga CSV dan feature_schema.json tersimpan di data/processed.
Berikutnya buka 04_modeling.ipynb.


## 7. Kesimpulan Feature Engineering

Tabel mingguan sekarang mempunyai input historis, target satu minggu ke depan, tanggal target, serta dua baseline. Sampel dipisahkan berdasarkan tanggal target, dan jumlah yang dapat dipakai bisa dibaca pada bagian 5.

Hal paling penting pada tahap ini bukan banyaknya fitur, melainkan ketepatan arah waktu. Model hanya boleh melihat informasi yang sudah ada ketika prediksi dibuat. Target kosong tidak diisi, jendela rolling harus lengkap, dan semua kandidat dibandingkan pada sampel yang sama.

**Cek pemahaman:** `head_t` adalah kondisi terakhir yang menjadi input; `target_head_m` adalah kondisi minggu depan yang ingin diprediksi. `sample_valid` hanya aturan kelengkapan, bukan tanda bahwa prediksi sudah benar. Belum ada model yang dilatih pada notebook ini.
